# 09 — Export a Colab-Sized Cache

Writes a thinned copy of one cache shard: the same 224px volumes, but only the
8 slices per series that fine-tuning uses anyway.

    full shard      1.2 MB/series     9.7 GB     will not transfer
    thinned shard   401 KB/series     3.2 GB     transfers fine

Pulling a full shard to Colab failed twice at ~5.5 GB with IncompleteRead, so
3.2 GB is the point of this. Nothing is lost that notebook 08 would have read:
it subsamples to these same 8 slices on load.

**No GPU.** This is a memmap read and a write, so it does not touch the weekly
GPU quota -- which matters when that quota is 25 hours and only one session
runs at a time.

**Attach:** one cache dataset, plus `knee-labels` and `knee_side_classifier` so
the whole Colab bundle comes out of one publish.

## Setup

In [ ]:
# Materialise the repo modules. Generated from source by
# tools/make_notebook.py -- do not edit here, edit src/ and regenerate.
import importlib, pathlib, sys

FILES = {
    'src/__init__.py': '',
    'src/config.py': '"""Central configuration: label names, cache geometry, and environment paths.\n\nEverything here is imported by both the training pipeline and the submission\nnotebook, so it must stay dependency-free apart from the standard library.\n"""\n\nimport os\nfrom pathlib import Path\n\n# ---------------------------------------------------------------------------\n# Labels\n# ---------------------------------------------------------------------------\n# Exact submission spelling. The apostrophe in "Baker\'s" and the spaces are\n# part of the column names -- do not normalise them anywhere that touches a\n# submission file.\nLABELS = [\n    "ACL",\n    "MCL",\n    "Medial Meniscus",\n    "Lateral Meniscus",\n    "Medial OA",\n    "Lateral OA",\n    "PF OA",\n    "Effusion",\n    "Synovitis",\n    "Baker\'s",\n    "Contusion",\n    "Fracture",\n]\n\nID_COL = "StudyInstanceUID"\nSUBMISSION_COLUMNS = [ID_COL] + LABELS\n\n# Compartment-specific labels. A horizontal flip swaps medial and lateral, so\n# any augmentation that mirrors the image must also swap these pairs.\nLATERAL_LABEL_PAIRS = [\n    ("Medial Meniscus", "Lateral Meniscus"),\n    ("Medial OA", "Lateral OA"),\n]\n\n# ---------------------------------------------------------------------------\n# Cache geometry\n# ---------------------------------------------------------------------------\n# Kept as constants so a higher-resolution variant is a config change rather\n# than a rewrite. Sizing: 224 * 224 * 24 bytes ~= 1.2 MB/series,\n# x 24,371 series ~= 30 GB, which fits Kaggle\'s 200 GB private dataset limit.\nCACHE_SIZE = 224          # in-plane resolution after resampling\nCACHE_SLICES = 24         # slices sampled per series, centred on the volume\nCACHE_DTYPE = "uint8"\n\n# Robust intensity normalisation percentiles. MRI has no standardised units,\n# so every volume is scaled against its own distribution.\nCLIP_PCT_LOW = 1.0\nCLIP_PCT_HIGH = 99.0\n\n# Canonical laterality: left knees are mirrored to match right knees so the\n# medial compartment always falls on the same side of the image.\nCANONICAL_SIDE = "R"\n\n# Mirroring is applied at load time, not baked into the cache.\n#\n# The side tag is blank on most series in this dataset and the series\n# descriptions are anonymised, so roughly 42% of series have no recoverable\n# side from metadata alone. Freezing a mirror decision into 30 GB of cache\n# would mean rebuilding all of it every time side detection improves -- and it\n# is expected to improve, via study-level propagation and then a classifier\n# trained on the series whose side is known. The cache therefore stores volumes\n# in their acquired orientation with the side recorded alongside, and the data\n# loader flips on read, which costs nothing.\nMIRROR_IN_CACHE = False\n\n# ---------------------------------------------------------------------------\n# Paths\n# ---------------------------------------------------------------------------\nCOMPETITION = "rsna-knee-abnormality-detection"\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\n\n\n# What a competition data root must contain to be recognised.\n_MARKERS = ("train_series", "test_series")\n\n\ndef _detect_data_root() -> Path:\n    """Locate the competition data across Kaggle, Colab, and local runs.\n\n    An explicit KNEE_DATA_ROOT wins. Otherwise Kaggle\'s input directory is\n    searched by content rather than by name: the mount is not reliably at\n    /kaggle/input/<competition-slug>, and guessing the name silently falls\n    through to the local fallback, which then fails much later with a confusing\n    path in the message.\n    """\n    env = os.environ.get("KNEE_DATA_ROOT")\n    if env:\n        return Path(env)\n\n    kaggle_input = Path("/kaggle/input")\n    if kaggle_input.exists():\n        exact = kaggle_input / COMPETITION\n        if exact.exists():\n            return exact\n        # Search two levels: Kaggle mounts competition data under\n        # /kaggle/input/competitions/<slug>/ as well as directly under\n        # /kaggle/input/<slug>/, and which one you get is not predictable.\n        try:\n            for child in sorted(kaggle_input.iterdir()):\n                if not child.is_dir():\n                    continue\n                if any((child / m).exists() for m in _MARKERS):\n                    return child\n                for grandchild in sorted(child.iterdir()):\n                    if grandchild.is_dir() and any(\n                            (grandchild / m).exists() for m in _MARKERS):\n                        return grandchild\n        except OSError:\n            pass\n\n    return PROJECT_ROOT / "data"\n\n\nDATA_ROOT = _detect_data_root()\nMETA_DIR = PROJECT_ROOT / "data" / "meta"\nCACHE_DIR = Path(os.environ.get("KNEE_CACHE_DIR", PROJECT_ROOT / "data" / "cache"))\n\nTRAIN_CSV = META_DIR / "train.csv"\nTRAIN_SERIES_CSV = META_DIR / "train_series.csv"\nTEST_CSV = META_DIR / "test.csv"\nTEST_SERIES_CSV = META_DIR / "test_series.csv"\nSAMPLE_SUBMISSION_CSV = META_DIR / "sample_submission.csv"\n\nINDEX_PARQUET = META_DIR / "index.parquet"\n\nON_KAGGLE = Path("/kaggle").exists()\n',
    'src/data/__init__.py': '',
    'src/data/dicom_io.py': '"""DICOM reading, geometric slice ordering, and laterality resolution.\n\nTwo operations here are load-bearing and fail silently if done naively:\n\n1. **Slice ordering.** InstanceNumber is not reliable across the 19 contributing\n   institutions. Slices are ordered by projecting ImagePositionPatient onto the\n   slice normal derived from ImageOrientationPatient, which is defined by the\n   scanner geometry rather than by convention.\n\n2. **Laterality.** Four of the twelve labels are compartment-specific\n   (medial vs. lateral). In an image of a left knee the medial compartment sits\n   on the opposite side of the frame from a right knee, so left knees must be\n   mirrored into a canonical orientation before a model can learn the\n   distinction.\n"""\n\nimport re\nfrom pathlib import Path\nfrom typing import Iterable, List, Optional, Sequence\n\nimport numpy as np\nimport pydicom\n\n# Header fields pulled during the index scan. Kept explicit so the parquet\n# schema is stable and the scan stays header-only.\nHEADER_FIELDS = [\n    "StudyInstanceUID",\n    "SeriesInstanceUID",\n    "SOPInstanceUID",\n    "Rows",\n    "Columns",\n    "PixelSpacing",\n    "ImagePositionPatient",\n    "ImageOrientationPatient",\n    "SliceThickness",\n    "SpacingBetweenSlices",\n    "EchoTime",\n    "RepetitionTime",\n    "InversionTime",\n    "ScanningSequence",\n    "SequenceVariant",\n    "SeriesDescription",\n    "ProtocolName",\n    "BodyPartExamined",\n    "Manufacturer",\n    "ManufacturerModelName",\n    "MagneticFieldStrength",\n    "ImageLaterality",\n    "Laterality",\n    "InstanceNumber",\n]\n\n# DICOM DS/IS values carry their original string representation, which would\n# land in the index as object dtype. Cast these explicitly.\nNUMERIC_FIELDS = {\n    "Rows",\n    "Columns",\n    "SliceThickness",\n    "SpacingBetweenSlices",\n    "EchoTime",\n    "RepetitionTime",\n    "InversionTime",\n    "MagneticFieldStrength",\n    "InstanceNumber",\n}\n\n_LEFT_PATTERN = re.compile(\n    r"\\b(l|lt|left|links|izquierda|izq|gauche|sinistra|sin|links?e)\\b",\n    re.IGNORECASE,\n)\n_RIGHT_PATTERN = re.compile(\n    r"\\b(r|rt|right|rechts|derecha|der|droite|destra|dx)\\b",\n    re.IGNORECASE,\n)\n\n\ndef read_header(path) -> dict:\n    """Read one DICOM\'s headers without decoding pixel data."""\n    ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)\n    row = {"path": str(path)}\n    for field in HEADER_FIELDS:\n        value = getattr(ds, field, None)\n        if value is None:\n            row[field] = None\n        elif isinstance(value, pydicom.multival.MultiValue):\n            # Geometry fields are numeric; ScanningSequence and friends are\n            # multi-valued strings, so fall back to a joined token.\n            try:\n                row[field] = [float(v) for v in value]\n            except (TypeError, ValueError):\n                row[field] = "\\\\".join(str(v) for v in value)\n        elif field in NUMERIC_FIELDS:\n            try:\n                row[field] = float(value)\n            except (TypeError, ValueError):\n                row[field] = None\n        elif isinstance(value, (int, float, str)):\n            row[field] = str(value) if isinstance(value, str) else value\n        else:\n            row[field] = str(value)\n    return row\n\n\ndef slice_normal(orientation: Optional[Sequence[float]]) -> np.ndarray:\n    """Unit normal of the image plane, from the two direction cosines.\n\n    Falls back to the z axis when orientation is missing, which keeps a\n    malformed series ordered by patient-superior position rather than crashing.\n    """\n    if orientation is None or len(orientation) != 6:\n        return np.array([0.0, 0.0, 1.0])\n\n    row_cosine = np.asarray(orientation[:3], dtype=float)\n    col_cosine = np.asarray(orientation[3:], dtype=float)\n    normal = np.cross(row_cosine, col_cosine)\n\n    magnitude = np.linalg.norm(normal)\n    if magnitude == 0:\n        return np.array([0.0, 0.0, 1.0])\n    return normal / magnitude\n\n\ndef slice_position(header: dict) -> float:\n    """Scalar position of a slice along its own stack axis."""\n    position = header.get("ImagePositionPatient")\n    if position is None or len(position) != 3:\n        # No geometry available; fall back to InstanceNumber so ordering is at\n        # least deterministic. The index scan flags these series separately.\n        number = header.get("InstanceNumber")\n        return float(number) if number is not None else 0.0\n\n    normal = slice_normal(header.get("ImageOrientationPatient"))\n    return float(np.dot(np.asarray(position, dtype=float), normal))\n\n\ndef sort_slices(headers: Iterable[dict]) -> List[dict]:\n    """Order a series\' slices along the acquisition axis."""\n    return sorted(headers, key=slice_position)\n\n\ndef has_geometry(headers: Sequence[dict]) -> bool:\n    """True when every slice carries the position and orientation tags."""\n    for header in headers:\n        position = header.get("ImagePositionPatient")\n        orientation = header.get("ImageOrientationPatient")\n        if position is None or len(position) != 3:\n            return False\n        if orientation is None or len(orientation) != 6:\n            return False\n    return True\n\n\ndef resolve_laterality(headers: Sequence[dict]) -> Optional[str]:\n    """Determine whether a series images a left or right knee.\n\n    Prefers the explicit DICOM tags, then falls back to parsing the free-text\n    description fields, which carry side markers in several languages. Returns\n    "L", "R", or None when the side cannot be established -- callers must treat\n    None as unusable rather than guessing, since a wrong side silently swaps the\n    medial and lateral labels.\n    """\n    for header in headers:\n        for field in ("ImageLaterality", "Laterality"):\n            value = header.get(field)\n            if isinstance(value, str):\n                # The tag holds "L"/"R" but also "LEFT"/"RIGHT", and is far more\n                # often present-but-empty than absent. An empty string is not a\n                # side and must not be read as one.\n                token = value.strip().upper()\n                if token in ("L", "LEFT"):\n                    return "L"\n                if token in ("R", "RIGHT"):\n                    return "R"\n\n    for field in ("SeriesDescription", "ProtocolName", "BodyPartExamined"):\n        for header in headers:\n            text = header.get(field)\n            if not isinstance(text, str):\n                continue\n            # Split on non-letters so "MR_Knie_Rechts" yields usable tokens.\n            spaced = re.sub(r"[^A-Za-zÀ-ÿ]+", " ", text)\n            left = bool(_LEFT_PATTERN.search(spaced))\n            right = bool(_RIGHT_PATTERN.search(spaced))\n            # Ambiguous strings mentioning both sides tell us nothing.\n            if left and not right:\n                return "L"\n            if right and not left:\n                return "R"\n\n    return None\n\n\ndef load_series_volume(paths: Sequence[str]) -> np.ndarray:\n    """Decode an ordered list of DICOM paths into a (slices, rows, cols) array.\n\n    Applies the rescale slope and intercept when present. Slices whose in-plane\n    shape differs from the first are skipped rather than silently broadcast.\n    """\n    slices = []\n    reference_shape = None\n    skipped = 0\n\n    for path in paths:\n        try:\n            ds = pydicom.dcmread(str(path), force=True)\n            pixels = ds.pixel_array.astype(np.float32)\n        except Exception:  # noqa: BLE001\n            # A truncated or malformed file loses one slice, not the series.\n            # The dataset contains files whose declared pixel length exceeds\n            # what is actually stored, and dropping 23 good slices over one\n            # bad one costs a whole study its imaging.\n            skipped += 1\n            continue\n\n        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)\n        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)\n        if slope != 1.0 or intercept != 0.0:\n            pixels = pixels * slope + intercept\n\n        if reference_shape is None:\n            reference_shape = pixels.shape\n        elif pixels.shape != reference_shape:\n            skipped += 1\n            continue\n\n        slices.append(pixels)\n\n    if not slices:\n        raise ValueError(f"no decodable slices in series ({skipped} unreadable)")\n\n    return np.stack(slices, axis=0)\n\n\ndef series_paths(series_dir) -> List[str]:\n    """All DICOM files under one series directory."""\n    return sorted(str(p) for p in Path(series_dir).glob("*.dcm"))\n\n\ndef resolve_study_laterality(series_sides: dict) -> dict:\n    """Fill in unknown series sides from other series in the same study.\n\n    A knee MRI study images one knee, so a side established on any series\n    applies to all of them. This matters because the DICOM side tags are\n    populated inconsistently across the contributing institutions -- a study\n    can carry the tag on its coronal series and omit it on the sagittals.\n\n    Report text is deliberately not used as a source here. Reports exist only\n    for training studies, so recovering a side from them would mirror the\n    training volumes and leave the test volumes untouched -- a train/test skew\n    that is worse than the missing side it fixes.\n\n    Args:\n        series_sides: {study_uid: {series_uid: "L" | "R" | None}}\n\n    Returns:\n        The same structure with unknown entries filled where the study allows.\n        Studies whose series disagree are left untouched, since a conflict\n        means the assumption of one knee per study does not hold there.\n    """\n    resolved = {}\n    for study_uid, series in series_sides.items():\n        known = {side for side in series.values() if side in ("L", "R")}\n\n        if len(known) == 1:\n            side = known.pop()\n            resolved[study_uid] = {uid: side for uid in series}\n        else:\n            # No evidence, or contradictory evidence. Guessing would silently\n            # swap the medial and lateral labels on four of twelve findings.\n            resolved[study_uid] = dict(series)\n\n    return resolved\n\n\ndef laterality_coverage(series_sides: dict) -> dict:\n    """Counts before and after study-level propagation, for auditing."""\n    before = sum(1 for s in series_sides.values() for v in s.values()\n                 if v in ("L", "R"))\n    total = sum(len(s) for s in series_sides.values())\n\n    filled = resolve_study_laterality(series_sides)\n    after = sum(1 for s in filled.values() for v in s.values()\n                if v in ("L", "R"))\n\n    conflicts = sum(\n        1 for s in series_sides.values()\n        if len({v for v in s.values() if v in ("L", "R")}) > 1\n    )\n\n    return {\n        "total_series": total,\n        "known_before": before,\n        "known_after": after,\n        "recovered": after - before,\n        "conflicting_studies": conflicts,\n    }\n',
    'src/data/cache.py': '"""Turn a raw DICOM series into a compact, model-ready uint8 volume.\n\nThe same code path runs during cache construction and at submission time. Any\ndivergence between the two produces a train/test preprocessing skew, which is\ninvisible in local validation and only shows up as a collapsed leaderboard\nscore -- so callers must import from here rather than reimplementing.\n\nOutput per series: (CACHE_SLICES, CACHE_SIZE, CACHE_SIZE) uint8, in the\norientation it was acquired in. Mirroring to a canonical side happens at load\ntime rather than here -- see config.MIRROR_IN_CACHE for why.\n"""\n\nfrom typing import Dict, List, Optional, Sequence\n\nimport cv2\nimport numpy as np\n\nfrom src import config\nfrom src.data import dicom_io\n\n\ndef normalize_intensity(volume: np.ndarray) -> np.ndarray:\n    """Scale a volume to uint8 using its own robust percentile range.\n\n    MRI carries no standardised units -- intensity depends on scanner, field\n    strength, coil, and whether fat suppression was applied -- so each volume is\n    scaled against its own distribution rather than a global constant.\n    """\n    finite = volume[np.isfinite(volume)]\n    if finite.size == 0:\n        return np.zeros(volume.shape, dtype=np.uint8)\n\n    low, high = np.percentile(finite, [config.CLIP_PCT_LOW, config.CLIP_PCT_HIGH])\n    if not np.isfinite(low) or not np.isfinite(high) or high <= low:\n        # Flat or degenerate volume; nothing to stretch.\n        return np.zeros(volume.shape, dtype=np.uint8)\n\n    scaled = (np.clip(volume, low, high) - low) / (high - low)\n    return (scaled * 255.0).round().astype(np.uint8)\n\n\ndef resize_volume(volume: np.ndarray, size: int = None) -> np.ndarray:\n    """Resample every slice to size x size with area/linear interpolation."""\n    size = size or config.CACHE_SIZE\n    output = np.empty((volume.shape[0], size, size), dtype=volume.dtype)\n\n    for i, plane in enumerate(volume):\n        # Area interpolation avoids the aliasing that bilinear introduces when\n        # downsampling 800px acquisitions; linear is correct when upsampling.\n        interp = cv2.INTER_AREA if plane.shape[0] > size else cv2.INTER_LINEAR\n        output[i] = cv2.resize(plane, (size, size), interpolation=interp)\n\n    return output\n\n\ndef select_slices(count: int, target: int = None) -> List[int]:\n    """Indices of `target` slices spread evenly across a stack of `count`.\n\n    Short series are handled by repeating edge slices rather than zero-padding,\n    so the volume stays anatomically continuous.\n    """\n    target = target or config.CACHE_SLICES\n    if count <= 0:\n        raise ValueError("cannot select slices from an empty series")\n    if count == target:\n        return list(range(count))\n\n    positions = np.linspace(0, count - 1, target)\n    return [int(round(p)) for p in positions]\n\n\ndef mirror_axis(orientation: Optional[Sequence[float]]) -> int:\n    """Which array axis runs medial-to-lateral, from the scanner geometry.\n\n    Mirroring a knee means reflecting through the patient\'s left-right axis,\n    and which array axis that corresponds to depends on the acquisition plane:\n\n        coronal / axial   left-right lies in-plane  -> flip columns (axis 2)\n        sagittal          left-right is the slice axis -> reverse slices (axis 0)\n\n    A blanket column flip is wrong for sagittal series. It leaves medial and\n    lateral exactly where they were -- they are separated across slices, not\n    across the image -- and instead reflects anterior-posterior, moving the\n    patella to the back of the knee.\n\n    Returns 0, 1, or 2: the axis of a (slices, rows, cols) volume to reverse.\n    """\n    left_right = np.array([1.0, 0.0, 0.0])\n\n    if orientation is None or len(orientation) != 6:\n        # No geometry to reason from. Columns is the common case (coronal and\n        # axial together outnumber sagittal), but the caller should prefer not\n        # to mirror at all when geometry is missing.\n        return 2\n\n    width = np.asarray(orientation[:3], dtype=float)    # increasing column index\n    height = np.asarray(orientation[3:], dtype=float)   # increasing row index\n    normal = dicom_io.slice_normal(orientation)\n\n    alignments = [\n        (abs(float(normal @ left_right)), 0),\n        (abs(float(height @ left_right)), 1),\n        (abs(float(width @ left_right)), 2),\n    ]\n    return max(alignments)[1]\n\n\ndef canonicalize_side(volume: np.ndarray, side: Optional[str],\n                      orientation: Optional[Sequence[float]] = None) -> np.ndarray:\n    """Mirror left knees so the medial compartment is always on one side.\n\n    Four labels are compartment-specific (Medial/Lateral Meniscus, Medial/\n    Lateral OA). Without this the model would have to infer the side before it\n    could place a finding, from a cue that is not reliably in the image.\n\n    The axis reflected is chosen from the acquisition geometry -- see\n    mirror_axis -- because it differs between sagittal and coronal series.\n    """\n    if side is None:\n        # Unknown side: leave untouched and let the caller decide. Guessing\n        # would silently swap four of the twelve labels.\n        return volume\n    if side.upper() == config.CANONICAL_SIDE:\n        return volume\n\n    axis = mirror_axis(orientation)\n    return np.flip(volume, axis=axis).copy()\n\n\ndef build_series_volume(headers: Sequence[dict]) -> Dict:\n    """Full pipeline for one series: sort, decode, normalise, resize.\n\n    Mirroring is applied only when config.MIRROR_IN_CACHE is set, which it is\n    not by default. The recorded `laterality` is what the loader mirrors on.\n    Returns the volume plus the provenance needed to audit it later.\n    """\n    if not headers:\n        raise ValueError("no headers supplied")\n\n    ordered = dicom_io.sort_slices(headers)\n    first_header = ordered[0]\n    side = dicom_io.resolve_laterality(ordered)\n    geometry_ok = dicom_io.has_geometry(ordered)\n\n    indices = select_slices(len(ordered))\n    selected = [ordered[i] for i in indices]\n\n    raw = dicom_io.load_series_volume([h["path"] for h in selected])\n\n    # Slices skipped as unreadable leave the volume short of CACHE_SLICES, and\n    # the cache is a fixed-shape array. Re-select across whatever decoded so\n    # the depth is exact regardless of how many files were dropped.\n    if raw.shape[0] != config.CACHE_SLICES:\n        raw = raw[select_slices(raw.shape[0])]\n\n    resized = resize_volume(raw.astype(np.float32))\n    normalized = normalize_intensity(resized)\n\n    # Left as acquired by default; the loader mirrors using the recorded side.\n    # See config.MIRROR_IN_CACHE for why this is not baked in here.\n    volume = (canonicalize_side(normalized, side,\n                                first_header.get("ImageOrientationPatient"))\n              if config.MIRROR_IN_CACHE else normalized)\n\n    first = ordered[0]\n    return {\n        "volume": volume,\n        "StudyInstanceUID": first.get("StudyInstanceUID"),\n        "SeriesInstanceUID": first.get("SeriesInstanceUID"),\n        "n_slices_raw": len(ordered),\n        "laterality": side,\n        "mirrored": bool(config.MIRROR_IN_CACHE and side is not None\n                         and side.upper() != config.CANONICAL_SIDE),\n        "geometry_ok": geometry_ok,\n        "rows": int(first.get("Rows") or 0),\n        "cols": int(first.get("Columns") or 0),\n    }\n\n\ndef build_series_record(item):\n    """Picklable worker for a process pool: (series_uid, headers) -> result.\n\n    Defined at module level deliberately. A worker defined inside a notebook\n    cell cannot be pickled by name, so ProcessPoolExecutor fails under the\n    spawn start method and survives under fork only by accident. Keeping it\n    importable makes the cache build work the same way everywhere.\n\n    Failures are returned rather than raised: one unreadable series must not\n    end a ninety-minute run, and the manifest records what to investigate.\n    """\n    series_uid, headers = item\n    try:\n        result = build_series_volume(headers)\n        return series_uid, result["volume"], {\n            "SeriesInstanceUID": series_uid,\n            "StudyInstanceUID": result["StudyInstanceUID"],\n            "n_slices_raw": result["n_slices_raw"],\n            "laterality": result["laterality"],\n            "mirrored": result["mirrored"],\n            "geometry_ok": result["geometry_ok"],\n            "rows": result["rows"],\n            "cols": result["cols"],\n            "error": None,\n        }\n    except Exception as exc:  # noqa: BLE001 - recorded, not raised\n        study_uid = headers[0].get("StudyInstanceUID") if headers else None\n        return series_uid, None, {\n            "SeriesInstanceUID": series_uid,\n            "StudyInstanceUID": study_uid,\n            "n_slices_raw": len(headers) if headers else 0,\n            "laterality": None,\n            "mirrored": False,\n            "geometry_ok": False,\n            "rows": 0,\n            "cols": 0,\n            "error": str(exc)[:200],\n        }\n',
    'src/data/index.py': '"""Header-only scan of the DICOM tree into a single parquet index.\n\nRuns on Kaggle, where the competition data is mounted. Reads no pixel data, so\nthe cost is dominated by file-system traversal rather than decoding. The\nresulting index drives cache construction and lets series be filtered and\naudited without touching the 570 GB of imaging again.\n\nUsage (in a Kaggle notebook):\n\n    from src.data.index import build_index\n    build_index("train_series", out_path="/kaggle/working/index_train.parquet")\n"""\n\nimport argparse\nimport os\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom typing import Iterator, List, Optional\n\nimport pandas as pd\n\nfrom src import config\nfrom src.data import dicom_io\n\n\ndef iter_dicom_paths(root) -> Iterator[Path]:\n    """Yield every .dcm under root/study/series/ without materialising a list."""\n    root = Path(root)\n    for study_dir in sorted(root.iterdir()):\n        if not study_dir.is_dir():\n            continue\n        for series_dir in sorted(study_dir.iterdir()):\n            if not series_dir.is_dir():\n                continue\n            for path in sorted(series_dir.glob("*.dcm")):\n                yield path\n\n\ndef scan_paths(paths: List[Path], workers: int = 8) -> pd.DataFrame:\n    """Read headers for a batch of files in parallel.\n\n    Header reads are I/O bound, so threads are the right tool -- the GIL is\n    released during file reads and a process pool would only add overhead.\n    """\n    def safe_read(path):\n        try:\n            return dicom_io.read_header(path)\n        except Exception as exc:  # noqa: BLE001 - one bad file must not stop a scan\n            return {"path": str(path), "error": str(exc)}\n\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        rows = list(pool.map(safe_read, paths))\n\n    return pd.DataFrame(rows)\n\n\ndef build_index(\n    split: str = "train_series",\n    out_path: Optional[str] = None,\n    data_root=None,\n    workers: int = 8,\n    batch_size: int = 5000,\n    limit: Optional[int] = None,\n) -> pd.DataFrame:\n    """Scan one split\'s DICOM tree and write a parquet index.\n\n    Processes in batches so a long scan reports progress and so peak memory\n    stays flat regardless of how many files the split holds.\n    """\n    root = Path(data_root or config.DATA_ROOT) / split\n    if not root.exists():\n        raise FileNotFoundError(f"{root} does not exist")\n\n    try:\n        from tqdm.auto import tqdm\n    except ImportError:  # progress is a convenience, never a requirement\n        tqdm = None\n\n    frames = []\n    batch: List[Path] = []\n    scanned = 0\n\n    # The tree is walked lazily, so the total is unknown until the end. tqdm\n    # counts files without a total rather than forcing a full listing first,\n    # which on this dataset would mean stat-ing ~820,000 paths up front.\n    progress = tqdm(total=limit, unit="file", desc=f"scanning {split}") if tqdm else None\n\n    for path in iter_dicom_paths(root):\n        batch.append(path)\n        if limit is not None and len(batch) + scanned >= limit:\n            break\n        if len(batch) >= batch_size:\n            frames.append(scan_paths(batch, workers=workers))\n            scanned += len(batch)\n            if progress is not None:\n                progress.update(len(batch))\n                progress.set_postfix(series=int(frames[-1]["SeriesInstanceUID"].nunique()))\n            else:\n                print(f"  scanned {scanned} files", flush=True)\n            batch = []\n\n    if batch:\n        frames.append(scan_paths(batch, workers=workers))\n        scanned += len(batch)\n        if progress is not None:\n            progress.update(len(batch))\n\n    if progress is not None:\n        progress.close()\n\n    if not frames:\n        raise RuntimeError(f"no DICOM files found under {root}")\n\n    index = pd.concat(frames, ignore_index=True)\n    print(f"scanned {scanned} files across {index[\'SeriesInstanceUID\'].nunique()} series")\n\n    if "error" in index.columns:\n        failures = int(index["error"].notna().sum())\n        if failures:\n            print(f"WARNING: {failures} files failed to parse")\n\n    if out_path:\n        os.makedirs(Path(out_path).parent, exist_ok=True)\n        index.to_parquet(out_path, index=False)\n        print(f"wrote {out_path}")\n\n    return index\n\n\ndef summarize(index: pd.DataFrame) -> pd.DataFrame:\n    """Per-series roll-up: slice count, geometry completeness, laterality.\n\n    Surfaces the series that need attention -- missing geometry, unresolvable\n    side -- before an hour of cache construction is spent on them.\n    """\n    records = []\n    for series_uid, group in index.groupby("SeriesInstanceUID", sort=False):\n        headers = group.to_dict("records")\n        records.append({\n            "StudyInstanceUID": headers[0].get("StudyInstanceUID"),\n            "SeriesInstanceUID": series_uid,\n            "n_slices": len(headers),\n            "geometry_ok": dicom_io.has_geometry(headers),\n            "laterality": dicom_io.resolve_laterality(headers),\n            "rows": headers[0].get("Rows"),\n            "cols": headers[0].get("Columns"),\n        })\n\n    return pd.DataFrame(records)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--split", default="train_series")\n    parser.add_argument("--out", default=None)\n    parser.add_argument("--workers", type=int, default=8)\n    parser.add_argument("--limit", type=int, default=None)\n    args = parser.parse_args()\n\n    build_index(split=args.split, out_path=args.out,\n                workers=args.workers, limit=args.limit)\n\n\nif __name__ == "__main__":\n    main()\n',
    'src/model/__init__.py': '',
    'src/model/side_classifier.py': '"""Predict whether a cached series images a left or right knee.\n\nThe DICOM side tag is blank on ~41% of series and the descriptions are\nanonymised, so metadata cannot resolve them. Four of the twelve findings are\ncompartment-specific, and placing them requires knowing the side, so the\nremainder is predicted from the imaging itself.\n\nThis trains on the 14,403 series whose side is known and applies to the rest.\nBoth halves come from the same cache, so it behaves identically at submission\ntime -- unlike report text, which exists only for training studies.\n\n**Why the whole ordered stack rather than one representative slice.** For\ncoronal and axial acquisitions the left-right axis lies in-plane, and a single\nimage settles the side: the fibula sits laterally. For sagittal acquisitions\nthe left-right axis IS the slice axis, so individual slices look much the same\neither way -- what differs is the direction of travel through the volume, which\nruns medial-to-lateral for one side and lateral-to-medial for the other. A\nmodel shown one slice cannot see that. Slices are therefore fed as ordered\nchannels, which preserves it.\n"""\n\nfrom typing import Optional, Sequence\n\nimport numpy as np\n\ntry:\n    import torch\n    import torch.nn as nn\nexcept ImportError:  # the module is importable without torch for testing\n    torch = None\n    nn = object\n\nfrom src import config\n\n# Downsampled in-plane resolution. The side of a knee is a coarse, global cue,\n# so half resolution is ample and quadruples throughput.\nINPUT_SIZE = 112\n\nSIDES = ["L", "R"]\n\n\ndef prepare_batch(volumes: np.ndarray, size: int = INPUT_SIZE) -> np.ndarray:\n    """Cache volumes to model input: (N, slices, size, size) float32 in [0, 1].\n\n    Slices become channels, keeping their order, so the network can use the\n    medial-to-lateral direction that distinguishes sides on sagittal series.\n    """\n    import cv2\n\n    if volumes.ndim == 3:\n        volumes = volumes[None]\n\n    count, depth, _, _ = volumes.shape\n    output = np.empty((count, depth, size, size), dtype=np.float32)\n\n    for i in range(count):\n        for j in range(depth):\n            output[i, j] = cv2.resize(volumes[i, j], (size, size),\n                                      interpolation=cv2.INTER_AREA)\n\n    return output / 255.0\n\n\ndef flip_input(batch: np.ndarray, plane_axis: Sequence[int]) -> np.ndarray:\n    """Mirror a prepared batch along each item\'s medial-lateral axis.\n\n    Used for augmentation: mirroring an input flips its label, which doubles\n    the training set and forces the model to key on anatomy rather than on any\n    incidental left/right imbalance in the source institutions.\n\n    `plane_axis` gives the cache axis (0 for slices, 2 for columns) per item;\n    in the prepared batch slices are dimension 1 and columns dimension 3.\n    """\n    output = np.empty_like(batch)\n    for i, axis in enumerate(plane_axis):\n        output[i] = np.flip(batch[i], axis=0 if axis == 0 else 2)\n    return output\n\n\nif torch is None:\n    # Importable without torch so the array handling above can be tested on a\n    # machine that never trains. The network itself needs the real thing.\n    SideNet = None\nelse:\n  class SideNet(nn.Module):\n      """A small convolutional classifier over slice-channel input.\n\n      Trained from scratch deliberately. It is a coarse two-class problem with\n      14,000 labelled examples, so a pretrained backbone buys little -- and the\n      competition\'s stance on pretrained weights is still unconfirmed, which this\n      piece of the pipeline should not depend on.\n      """\n\n      def __init__(self, in_channels: int = None, width: int = 32):\n          super().__init__()\n          in_channels = in_channels or config.CACHE_SLICES\n\n          def block(inp, out, stride=2):\n              return nn.Sequential(\n                  nn.Conv2d(inp, out, 3, stride=stride, padding=1, bias=False),\n                  nn.BatchNorm2d(out),\n                  nn.ReLU(inplace=True),\n                  nn.Conv2d(out, out, 3, padding=1, bias=False),\n                  nn.BatchNorm2d(out),\n                  nn.ReLU(inplace=True),\n              )\n\n          self.features = nn.Sequential(\n              block(in_channels, width),        # 112 -> 56\n              block(width, width * 2),          # 56 -> 28\n              block(width * 2, width * 4),      # 28 -> 14\n              block(width * 4, width * 4),      # 14 -> 7\n          )\n          self.head = nn.Sequential(\n              nn.AdaptiveAvgPool2d(1),\n              nn.Flatten(),\n              nn.Dropout(0.2),\n              nn.Linear(width * 4, 1),\n          )\n\n      def forward(self, x):\n          return self.head(self.features(x)).squeeze(-1)\n\n\ndef study_level_vote(predictions, study_uids, confidences=None) -> dict:\n    """Collapse per-series predictions to one side per study.\n\n    A study is one knee, so its series must agree. Voting across them is\n    strictly more reliable than any single series -- and the observed\n    conflicting-tag rate is under 1%, so the assumption holds in practice.\n\n    Returns {study_uid: "L" | "R"}.\n    """\n    if len(predictions) != len(study_uids):\n        raise ValueError(\n            f"predictions and study_uids differ in length: "\n            f"{len(predictions)} vs {len(study_uids)}")\n    if confidences is None:\n        confidences = np.ones(len(predictions))\n    elif len(confidences) != len(predictions):\n        # zip would truncate silently, shifting weights onto the wrong series.\n        raise ValueError(\n            f"confidences length {len(confidences)} does not match "\n            f"{len(predictions)} predictions")\n\n    scores = {}\n    for prediction, study, weight in zip(predictions, study_uids, confidences):\n        scores.setdefault(study, []).append((float(prediction), float(weight)))\n\n    result = {}\n    for study, entries in scores.items():\n        total = sum(weight for _, weight in entries)\n        mean = (sum(p * w for p, w in entries) / total) if total else 0.5\n        result[study] = "R" if mean >= 0.5 else "L"\n    return result\n\n\ndef accuracy(predictions: np.ndarray, truth: np.ndarray) -> float:\n    """Fraction correct, with predictions as probabilities of the right knee."""\n    return float(((predictions >= 0.5).astype(int) == truth).mean())\n',
    'src/model/embed.py': '"""Frozen DINOv2 slice embeddings.\n\nRuns the backbone once over every cached slice and stores the result, so the\naggregation stack trains on cached features in minutes rather than hours. The\nsame code runs at submission time, which is the point -- a divergence between\nhow features are computed for training and for inference is invisible locally\nand fatal on the leaderboard.\n\n**Why frozen, and why first.** The competition scores a second prize track on\n`AUC + RuntimeSeconds / 32400`, so inference cost is a scored quantity rather\nthan a ceiling to stay under. A frozen trunk is cheap at submission and lets\nthe aggregation design -- which is where the modelling actually happens -- be\niterated dozens of times a day instead of three.\n\n**What is stored per slice.** The CLS token concatenated with the patch tokens\nreduced by `POOL_GRID`:\n\n    grid 1     CLS + mean of all patches      768 dims    0.9 GB\n    grid 2     CLS + 2x2 pooled grid        1,920 dims    2.3 GB\n    grid 4     CLS + 4x4 pooled grid        6,528 dims    7.6 GB\n\nStoring the full 16x16 grid would cost ~115 GB, which is why it is not offered.\n\nThe first model used grid 1, and its weakness appeared exactly where the mean\npredicts one: regional findings (effusion, OA, Baker\'s) averaged 0.865 on gold\nwhile focal ones (meniscus, ligaments, fracture) averaged 0.695. A meniscal\ntear occupies two or three patches out of 256, so a mean over all of them\nburies it in 250-odd patches of normal tissue.\n\nWide embeddings must be projected before the aggregation stack -- see\n`KneeNet(input_dim=...)`. Running the stack at 6,528 wide puts ~769M parameters\nin it against 8.2M with a projection.\n"""\n\nfrom typing import Optional\n\nimport numpy as np\n\ntry:\n    import torch\n    import torch.nn as nn\nexcept ImportError:\n    torch = None\n    nn = None\n\nfrom src import config\n\n# The registers variant. Registers exist to absorb the high-norm artefact\n# tokens that otherwise contaminate patch features -- precisely what gets\n# pooled here.\nMODEL_NAME = "vit_small_patch14_reg4_dinov2.lvd142m"\n\n# timm\'s DINOv2 defaults to 518. The cache is 224, which divides cleanly by the\n# patch size of 14 into a 16x16 grid, so the size is passed explicitly and the\n# position embeddings are interpolated rather than silently mismatched.\nINPUT_SIZE = config.CACHE_SIZE\n\n# DINOv2-S token width.\nTOKEN_DIM = 384\n\n# How the 16x16 patch grid is reduced before storage.\n#\n#   1  mean over all patches       768 dims   0.9 GB over the dataset\n#   4  4x4 average-pooled grid   6,528 dims   7.6 GB\n#\n# The mean is what the first model used, and its weakness showed up exactly\n# where predicted: regional findings averaged 0.865 on gold while focal ones\n# averaged 0.695. A meniscal tear occupies two or three patches out of 256, so\n# averaging buries it. A 4x4 grid keeps enough spatial structure for the\n# attention pooling to find it.\nPOOL_GRID = 1\n\n\ndef embed_dim(grid: int = None) -> int:\n    """Stored width per slice: CLS plus the pooled patch tokens."""\n    grid = POOL_GRID if grid is None else grid\n    tokens = 1 + (grid * grid if grid > 1 else 1)\n    return TOKEN_DIM * tokens\n\n\nEMBED_DIM = embed_dim()\n\n# ImageNet statistics, which DINOv2 was trained under.\nMEAN = (0.485, 0.456, 0.406)\nSTD = (0.229, 0.224, 0.225)\n\n\ndef build_encoder(model_name: str = MODEL_NAME, pretrained: bool = True,\n                  checkpoint_path: Optional[str] = None):\n    """A frozen DINOv2 feature extractor in eval mode.\n\n    `checkpoint_path` loads weights from disk, which the submission notebook\n    needs: internet is disabled there, so timm cannot fetch anything.\n    """\n    import timm\n\n    model = timm.create_model(\n        model_name,\n        pretrained=pretrained and checkpoint_path is None,\n        img_size=INPUT_SIZE,\n        num_classes=0,\n    )\n\n    if checkpoint_path:\n        state = torch.load(checkpoint_path, map_location="cpu")\n        state = state.get("state_dict", state)\n        model.load_state_dict(state, strict=False)\n\n    model.eval()\n    for parameter in model.parameters():\n        parameter.requires_grad = False\n    return model\n\n\ndef normalize(slices):\n    """uint8 (N, 3, H, W) -> normalised float tensor."""\n    x = slices.float().div_(255.0) if slices.dtype == torch.uint8 else slices.float()\n    mean = torch.tensor(MEAN, device=x.device).view(1, 3, 1, 1)\n    std = torch.tensor(STD, device=x.device).view(1, 3, 1, 1)\n    return (x - mean) / std\n\n\ndef pool_patches(patches: "torch.Tensor", grid: int) -> "torch.Tensor":\n    """(N, P, C) patch tokens -> (N, grid*grid*C), spatially pooled.\n\n    The patch sequence is reshaped back to its square grid before pooling, so\n    neighbouring patches are averaged with their neighbours rather than with\n    whatever happened to sit beside them in a flattened sequence.\n    """\n    import torch.nn.functional as F\n\n    count, n_patches, channels = patches.shape\n    side = int(round(n_patches ** 0.5))\n    if side * side != n_patches:\n        # Not a square grid; fall back to the mean rather than mis-shaping it.\n        return patches.mean(dim=1)\n\n    square = patches.transpose(1, 2).reshape(count, channels, side, side)\n    pooled = F.adaptive_avg_pool2d(square, grid)\n    return pooled.flatten(2).transpose(1, 2).reshape(count, grid * grid * channels)\n\n\ndef embed_slices(encoder, slices, grid: int = None) -> "torch.Tensor":\n    """(N, 3, H, W) -> (N, embed_dim(grid)): CLS plus pooled patch tokens.\n\n    Uses forward_features so the patch tokens are reachable at all; the model\'s\n    default pooled output discards them.\n    """\n    grid = POOL_GRID if grid is None else grid\n\n    with torch.no_grad():\n        tokens = encoder.forward_features(normalize(slices))\n\n        if tokens.ndim == 2:\n            # A backbone that already pools; repeat so the width is predictable.\n            repeats = embed_dim(grid) // tokens.shape[-1]\n            return tokens.repeat(1, repeats)\n\n        cls = tokens[:, 0]\n        prefix = getattr(encoder, "num_prefix_tokens", 1)\n        patches = tokens[:, prefix:]\n\n        if grid > 1:\n            return torch.cat([cls, pool_patches(patches, grid)], dim=-1)\n        return torch.cat([cls, patches.mean(dim=1)], dim=-1)\n\n\nclass CachedEmbeddingBackbone(nn.Module if nn is not None else object):\n    """Adapter so KneeNet can consume precomputed embeddings unchanged.\n\n    KneeNet expects a module mapping slices to features. When features are\n    already computed, the "backbone" is an identity that reshapes -- which\n    keeps one model definition for both the cached and the end-to-end path.\n    """\n\n    def __init__(self, feature_dim: int = EMBED_DIM):\n        super().__init__()\n        self.feature_dim = feature_dim\n\n    def forward(self, x):\n        return x\n',
    'src/model/dataset.py': '"""Study-level dataset over the cached volumes.\n\nJoins four sources into one training sample per study:\n\n    cache shards      the uint8 volumes, memory-mapped\n    manifest          which shard row holds which series\n    fused labels      12 soft targets and 12 sample weights per study\n    side predictions  L/R per series, from the tag or the classifier\n\nA sample is a whole study, not a series. The twelve findings are study-level\nlabels, and a single series often cannot support them -- effusion reads off a\nfluid-sensitive sequence, patellofemoral OA off a different one -- so the model\nhas to see the study\'s series together.\n\n**Laterality is handled two ways, depending on the path.**\n\nFor the end-to-end path, `mirror_volume` reflects left knees onto the canonical\nside at load time, along the axis the acquisition plane dictates: slice order\nfor sagittal, where medial-lateral is the slice axis, and columns for coronal\nand axial, where it lies in-plane. Doing it at load time rather than in the\ncache means side detection can improve without rebuilding 29 GB.\n\nFor the frozen path this is not available -- the embeddings are computed from\nthe volume, so the image can no longer be flipped. There the side travels to\nthe model as a covariate (`is_right`, `side_known`) instead. That still\naddresses the original concern, which was that the model should not have to\n*infer* the side from cues that are not reliably present; it is handed the\nanswer either way. Which of the two performs better is an experiment, not a\nsettled question.\n"""\n\nfrom typing import Dict, List, Optional, Sequence\n\nimport numpy as np\nimport pandas as pd\n\nfrom src import config\n\n# Plane -> which axis of a (slices, rows, cols) volume runs medial-to-lateral.\nPLANE_MIRROR_AXIS = {\n    "Sagittal": 0,   # medial-lateral is the slice axis\n    "Coronal": 2,    # medial-lateral lies in-plane\n    "Axial": 2,\n}\nDEFAULT_MIRROR_AXIS = 2\n\n# Series metadata fed to the model as conditioning. Findings are\n# sequence-specific, and these three are given in both train and test.\n# `is_right` and `side_known` carry laterality as a covariate rather than\n# normalising it away by mirroring. With precomputed frozen embeddings the\n# image can no longer be flipped -- the features are already computed -- so the\n# side is handed to the model instead, which addresses the original concern\n# (the model should not have to *infer* the side) by a different route.\n# Mirroring remains available for the end-to-end path and is an A/B to run,\n# not a settled question.\nMETA_COLUMNS = ["Fluid_Sensitive", "Fat_Suppression", "is_sagittal",\n                "is_coronal", "is_axial", "is_right", "side_known"]\n\n\ndef mirror_volume(volume: np.ndarray, side: Optional[str],\n                  plane: Optional[str]) -> np.ndarray:\n    """Reflect a left knee onto the canonical side, along the correct axis."""\n    if side is None or (isinstance(side, float) and np.isnan(side)):\n        # Unknown side: leave as acquired. Guessing swaps medial with lateral\n        # on four of the twelve findings.\n        return volume\n    if str(side).upper() == config.CANONICAL_SIDE:\n        return volume\n\n    axis = PLANE_MIRROR_AXIS.get(str(plane), DEFAULT_MIRROR_AXIS)\n    return np.flip(volume, axis=axis).copy()\n\n\ndef series_metadata(row: pd.Series) -> np.ndarray:\n    """Conditioning vector for one series.\n\n    Fluid sensitivity and fat suppression are kept separate: the data\n    description notes they are often correlated but not equivalent, so\n    collapsing them would discard the cases where they disagree.\n    """\n    plane = str(row.get("Anatomical_Plane", ""))\n    side = row.get("side_predicted")\n    known = isinstance(side, str) and side.upper() in ("L", "R")\n    return np.array([\n        float(row.get("Fluid_Sensitive", 0) or 0),\n        float(row.get("Fat_Suppression", 0) or 0),\n        float(plane == "Sagittal"),\n        float(plane == "Coronal"),\n        float(plane == "Axial"),\n        float(known and side.upper() == "R"),\n        float(known),\n    ], dtype=np.float32)\n\n\nclass StudyIndex:\n    """Everything needed to assemble a study, resolved once up front.\n\n    Built from frames rather than files so it can be constructed in a test\n    without touching disk.\n    """\n\n    def __init__(self, manifest: pd.DataFrame, series_meta: pd.DataFrame,\n                 labels: Optional[pd.DataFrame] = None,\n                 sides: Optional[pd.DataFrame] = None,\n                 max_series: Optional[int] = None):\n        frame = manifest.merge(\n            series_meta[["SeriesInstanceUID", "Fluid_Sensitive",\n                         "Fat_Suppression", "Anatomical_Plane"]],\n            on="SeriesInstanceUID", how="left")\n\n        if sides is not None:\n            frame = frame.merge(\n                sides[["SeriesInstanceUID", "side_predicted"]],\n                on="SeriesInstanceUID", how="left")\n        else:\n            frame["side_predicted"] = frame.get("laterality_resolved")\n\n        # Drop series whose volume failed to build; their row in the cache is\n        # zeros and would train the model on blank images.\n        if "error" in frame.columns:\n            frame = frame[frame["error"].isna()].reset_index(drop=True)\n\n        self.series = frame\n        self.max_series = max_series\n        self.labels = labels\n\n        self.studies: List[str] = sorted(frame[config.ID_COL].unique())\n        self._by_study = {\n            study: group.reset_index(drop=True)\n            for study, group in frame.groupby(config.ID_COL)\n        }\n\n        if labels is not None:\n            keyed = labels.set_index(config.ID_COL)\n            self.studies = [s for s in self.studies if s in keyed.index]\n            self._labels = keyed\n        else:\n            self._labels = None\n\n    def __len__(self) -> int:\n        return len(self.studies)\n\n    def series_for(self, study: str) -> pd.DataFrame:\n        """Series of a study, capped deterministically when a cap is set.\n\n        The cap exists for the efficiency track: inference cost scales with\n        series count, and a study with a dozen series contributes far more\n        runtime than one with three. Selection is by position rather than at\n        random so a prediction is reproducible.\n        """\n        rows = self._by_study[study]\n        if self.max_series is not None and len(rows) > self.max_series:\n            keep = np.linspace(0, len(rows) - 1, self.max_series).round().astype(int)\n            rows = rows.iloc[np.unique(keep)].reset_index(drop=True)\n        return rows\n\n    def targets_for(self, study: str):\n        """(targets, weights) for a study, or (None, None) when unlabelled."""\n        if self._labels is None or study not in self._labels.index:\n            return None, None\n        row = self._labels.loc[study]\n        targets = np.array([float(row[name]) for name in config.LABELS],\n                           dtype=np.float32)\n        weights = np.array(\n            [float(row.get(f"{name}_weight", 1.0)) for name in config.LABELS],\n            dtype=np.float32)\n        return targets, weights\n\n\ndef load_study(index: StudyIndex, study: str, shards: Dict[str, np.ndarray]) -> Dict:\n    """Assemble one study: mirrored volumes, conditioning, targets, weights."""\n    rows = index.series_for(study)\n\n    volumes, meta = [], []\n    for _, row in rows.iterrows():\n        shard = shards[row["cache_path"]]\n        volume = np.asarray(shard[int(row["position"])])\n        volume = mirror_volume(volume, row.get("side_predicted"),\n                               row.get("Anatomical_Plane"))\n        volumes.append(volume)\n        meta.append(series_metadata(row))\n\n    targets, weights = index.targets_for(study)\n    return {\n        config.ID_COL: study,\n        "volumes": np.stack(volumes) if volumes else np.zeros(\n            (0, config.CACHE_SLICES, config.CACHE_SIZE, config.CACHE_SIZE),\n            dtype=np.uint8),\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\n                                                     dtype=np.float32),\n        "targets": targets,\n        "weights": weights,\n    }\n\n\ndef load_study_features(index: StudyIndex, study: str,\n                        shards: Dict[str, np.ndarray]) -> Dict:\n    """Assemble one study from precomputed slice embeddings.\n\n    Mirroring is unavailable here -- the features were computed from the image\n    and cannot be flipped after the fact -- so laterality reaches the model\n    through `series_metadata` instead.\n    """\n    rows = index.series_for(study)\n\n    features, meta = [], []\n    for _, row in rows.iterrows():\n        shard = shards[row["cache_path"]]\n        features.append(np.asarray(shard[int(row["position"])], dtype=np.float32))\n        meta.append(series_metadata(row))\n\n    targets, weights = index.targets_for(study)\n    return {\n        config.ID_COL: study,\n        "volumes": np.stack(features) if features else np.zeros((0, 1, 1),\n                                                                dtype=np.float32),\n        "meta": np.stack(meta) if meta else np.zeros((0, len(META_COLUMNS)),\n                                                     dtype=np.float32),\n        "targets": targets,\n        "weights": weights,\n    }\n\n\ndef collate_studies(batch: Sequence[Dict]) -> Dict:\n    """Pad a batch of studies to the largest series count, with a mask.\n\n    Series counts vary from one to a dozen, so the batch is padded and masked\n    rather than truncated. The mask must reach the attention pooling: without\n    it, padding rows are averaged in as if they were real series.\n    """\n    import torch\n\n    longest = max(len(item["volumes"]) for item in batch)\n\n    # Per-series shape is taken from the data rather than assumed, so the same\n    # collate serves raw volumes (D, H, W) and cached embeddings (D, F).\n    reference = next((item["volumes"] for item in batch if len(item["volumes"])),\n                     None)\n    if reference is None:\n        raise ValueError("every study in the batch is empty")\n    per_series = reference.shape[1:]\n    dtype = reference.dtype\n\n    volumes = np.zeros((len(batch), longest, *per_series), dtype=dtype)\n    meta = np.zeros((len(batch), longest, len(META_COLUMNS)), dtype=np.float32)\n    mask = np.zeros((len(batch), longest), dtype=bool)\n\n    for i, item in enumerate(batch):\n        count = len(item["volumes"])\n        if count:\n            volumes[i, :count] = item["volumes"]\n            meta[i, :count] = item["meta"]\n            mask[i, :count] = True\n\n    out = {\n        config.ID_COL: [item[config.ID_COL] for item in batch],\n        "volumes": torch.from_numpy(volumes),\n        "meta": torch.from_numpy(meta),\n        "mask": torch.from_numpy(mask),\n    }\n\n    if batch[0]["targets"] is not None:\n        out["targets"] = torch.from_numpy(\n            np.stack([item["targets"] for item in batch]))\n        out["weights"] = torch.from_numpy(\n            np.stack([item["weights"] for item in batch]))\n    return out\n',
    'src/model/net.py': '"""Study-level model: slices to series to study to twelve logits.\n\nThe backbone is deliberately the least interesting part. It is swappable, and\nwhich one wins is settled by measurement rather than by argument -- at 4,349\nstudies with mined labels, the aggregation above it matters more.\n\n    (B, S, D, H, W) uint8 volumes\n      -> backbone over slices          (B*S*D, F)\n      -> attention pool over slices    (B*S, F)\n      -> FiLM on sequence metadata     (B*S, F)\n      -> transformer over series       (B, F)      masked\n      -> head                          (B, 12)\n\n**Why FiLM on the sequence metadata.** Findings are sequence-specific: effusion\nand Baker\'s cysts read off fluid-sensitive series, cartilage loss off\nfat-suppressed ones, ACL tears off sagittals. `Fluid_Sensitive`,\n`Fat_Suppression`, and the plane are given in both train and test, so pooling\nevery series through one undifferentiated path throws away free signal.\n\n**Why masking is load-bearing.** Series counts run from one to a dozen. Padded\nrows must be excluded from both the attention pool and the transformer, or a\nshort study is diluted by rows of zeros that the model reads as real series.\n"""\n\nfrom typing import Optional\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom src import config\nfrom src.model.dataset import META_COLUMNS\n\n# Adjacent slices stacked into the three channels an ImageNet-pretrained\n# backbone expects. Gives each slice a little through-plane context for free.\nSLICE_CONTEXT = 3\n\n\ndef stack_2p5d(volumes: torch.Tensor) -> torch.Tensor:\n    """(N, D, H, W) -> (N*D, 3, H, W) using each slice and its neighbours.\n\n    Edge slices repeat, which is preferable to zero padding: a black plane at\n    the edge of a volume is a feature the model would learn to key on.\n    """\n    n, depth, height, width = volumes.shape\n    previous = torch.cat([volumes[:, :1], volumes[:, :-1]], dim=1)\n    following = torch.cat([volumes[:, 1:], volumes[:, -1:]], dim=1)\n    stacked = torch.stack([previous, volumes, following], dim=2)\n    return stacked.reshape(n * depth, SLICE_CONTEXT, height, width)\n\n\nclass AttentionPool(nn.Module):\n    """Masked attention pooling over a sequence of embeddings.\n\n    Preferred to mean pooling because the informative slices are a minority --\n    a meniscal tear appears on a few slices of a stack of 24, and averaging\n    dilutes it by the rest.\n    """\n\n    def __init__(self, dim: int, hidden: int = 128):\n        super().__init__()\n        self.score = nn.Sequential(\n            nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))\n\n    def forward(self, x: torch.Tensor,\n                mask: Optional[torch.Tensor] = None) -> torch.Tensor:\n        # x: (B, N, F), mask: (B, N) with True for real entries.\n        weights = self.score(x).squeeze(-1)\n        if mask is not None:\n            weights = weights.masked_fill(~mask, float("-inf"))\n        weights = torch.softmax(weights, dim=1).unsqueeze(-1)\n        return (x * weights).sum(dim=1)\n\n\nclass FiLM(nn.Module):\n    """Feature-wise modulation of series embeddings by sequence metadata."""\n\n    def __init__(self, dim: int, meta_dim: int = len(META_COLUMNS)):\n        super().__init__()\n        self.to_scale_shift = nn.Sequential(\n            nn.Linear(meta_dim, dim), nn.ReLU(inplace=True),\n            nn.Linear(dim, dim * 2))\n\n    def forward(self, x: torch.Tensor, meta: torch.Tensor) -> torch.Tensor:\n        scale, shift = self.to_scale_shift(meta).chunk(2, dim=-1)\n        # Centre the scale on 1 so an untrained FiLM is close to the identity.\n        return x * (1.0 + scale) + shift\n\n\nclass KneeNet(nn.Module):\n    """Backbone plus the aggregation stack, emitting twelve study-level logits.\n\n    `backbone` is any module mapping (N, 3, H, W) to (N, feature_dim). Passing\n    a frozen encoder is supported and is the intended first configuration: with\n    the trunk frozen, slice embeddings are computed once and the stack above\n    trains in minutes rather than hours.\n    """\n\n    def __init__(self, backbone: Optional[nn.Module], feature_dim: int,\n                 n_labels: int = None, series_layers: int = 2,\n                 n_heads: int = 8, dropout: float = 0.2,\n                 freeze_backbone: bool = False,\n                 input_dim: Optional[int] = None):\n        super().__init__()\n        # None is valid on the frozen path, where the trunk ran offline.\n        self.backbone = backbone if backbone is not None else nn.Identity()\n        self.feature_dim = feature_dim\n        n_labels = n_labels or len(config.LABELS)\n\n        if freeze_backbone:\n            for parameter in self.backbone.parameters():\n                parameter.requires_grad = False\n\n        # Spatially pooled embeddings are wide (6,528 for a 4x4 grid), and\n        # running the whole stack at that width would put ~85M parameters in\n        # FiLM alone. A projection keeps the aggregation at a sane size.\n        self.project = (nn.Linear(input_dim, feature_dim)\n                        if input_dim and input_dim != feature_dim\n                        else nn.Identity())\n\n        self.slice_pool = AttentionPool(feature_dim)\n        self.film = FiLM(feature_dim)\n\n        layer = nn.TransformerEncoderLayer(\n            d_model=feature_dim, nhead=n_heads,\n            dim_feedforward=feature_dim * 2, dropout=dropout,\n            batch_first=True, norm_first=True)\n        self.series_encoder = nn.TransformerEncoder(layer, num_layers=series_layers)\n        self.study_pool = AttentionPool(feature_dim)\n\n        self.head = nn.Sequential(\n            nn.LayerNorm(feature_dim),\n            nn.Dropout(dropout),\n            nn.Linear(feature_dim, n_labels))\n\n    def encode_series(self, volumes: torch.Tensor) -> torch.Tensor:\n        """(N, D, H, W) uint8 or float -> (N, F) one embedding per series."""\n        count, depth = volumes.shape[0], volumes.shape[1]\n        if volumes.dtype == torch.uint8:\n            volumes = volumes.float().div_(255.0)\n\n        slices = stack_2p5d(volumes)\n        features = self.backbone(slices)\n        features = features.reshape(count, depth, self.feature_dim)\n        return self.slice_pool(features)\n\n    def aggregate(self, series: torch.Tensor, meta: torch.Tensor,\n                  mask: torch.Tensor) -> torch.Tensor:\n        """(B, S, F) series embeddings -> (B, n_labels).\n\n        Everything above the backbone. Shared by the end-to-end path and the\n        frozen path so the two cannot drift into different architectures.\n        """\n        series = self.film(series, meta)\n\n        # The transformer takes True for positions to IGNORE, the inverse of\n        # our mask.\n        encoded = self.series_encoder(series, src_key_padding_mask=~mask)\n        # A fully padded row emits NaN from the attention; zero it before\n        # pooling so one short study cannot poison the batch.\n        encoded = torch.nan_to_num(encoded)\n\n        pooled = self.study_pool(encoded, mask=mask)\n        return self.head(pooled)\n\n    def forward_features(self, features: torch.Tensor, meta: torch.Tensor,\n                         mask: torch.Tensor) -> torch.Tensor:\n        """Frozen path: (B, S, D, F) precomputed slice embeddings.\n\n        The backbone has already run, so this starts at the slice pooling.\n        """\n        batch, series, depth, _ = features.shape\n        projected = self.project(features.reshape(batch * series, depth, -1))\n        pooled = self.slice_pool(projected)\n        return self.aggregate(pooled.reshape(batch, series, self.feature_dim),\n                              meta, mask)\n\n    def forward(self, volumes: torch.Tensor, meta: torch.Tensor,\n                mask: torch.Tensor) -> torch.Tensor:\n        # volumes: (B, S, D, H, W), meta: (B, S, M), mask: (B, S)\n        batch, series = volumes.shape[0], volumes.shape[1]\n\n        flat = volumes.reshape(batch * series, *volumes.shape[2:])\n        embeddings = self.encode_series(flat).reshape(batch, series, self.feature_dim)\n        return self.aggregate(embeddings, meta, mask)\n\n\ndef weighted_bce(logits: torch.Tensor, targets: torch.Tensor,\n                 weights: torch.Tensor) -> torch.Tensor:\n    """BCE against soft targets, weighted per study and finding.\n\n    Targets are probabilities rather than 0/1 -- they come from two extractors\n    that disagree often enough that thresholding would teach the model to\n    reproduce extraction errors. Weights carry how much each target is\n    believed: full for agreement, less for conflict, least for silence.\n    """\n    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")\n    total = weights.sum()\n    if total <= 0:\n        return loss.mean() * 0.0\n    return (loss * weights).sum() / total\n',
    'src/model/splits.py': '"""Cross-validation splits and the gold holdout.\n\nTwo rules govern every split here, and both exist because the obvious\nalternative produces a number that looks better and means less.\n\n**Split by study, never by series.** A study\'s ~5.5 series image the same knee\nin the same session. A random series split puts near-duplicates on both sides\nand reports an accuracy the model does not have.\n\n**The 58 gold studies never enter training.** They are the only radiologist\nlabels in the dataset and the only honest measurement available. Everything\nelse is trained on mined labels, and a model evaluated against mined labels can\nscore well by reproducing the extractor\'s mistakes. `assert_no_gold_leakage`\nis called by the training entrypoint rather than trusted to discipline.\n"""\n\nfrom typing import Iterator, List, Optional, Sequence, Tuple\n\nimport numpy as np\nimport pandas as pd\n\nfrom src import config\n\nN_FOLDS = 5\nSEED = 0\n\n\ndef load_gold_uids(train_csv=None) -> set:\n    """Study UIDs of the 58 radiologist-labelled studies."""\n    frame = pd.read_csv(train_csv or config.TRAIN_CSV)\n    labelled = frame[frame[config.LABELS[0]].notna()]\n    return set(labelled[config.ID_COL])\n\n\ndef assert_no_gold_leakage(train_uids: Sequence[str], gold_uids: set) -> None:\n    """Fail loudly if a gold study appears in training data.\n\n    Called from the training entrypoint. The failure this guards against is\n    silent -- a leaked gold study inflates the one number being used to decide\n    whether the model works.\n    """\n    leaked = set(train_uids) & gold_uids\n    if leaked:\n        raise ValueError(\n            f"{len(leaked)} gold studies leaked into training: "\n            f"{sorted(leaked)[:3]}..."\n        )\n\n\ndef stratification_key(labels: pd.DataFrame) -> pd.Series:\n    """A coarse key for balancing folds across a 12-label problem.\n\n    Full multi-label stratification over 2^12 combinations would leave most\n    cells empty at 4,349 studies. The count of positive findings captures the\n    axis that matters -- studies with many findings are rare and should not\n    cluster in one fold.\n    """\n    positives = (labels[config.LABELS] > 0.5).sum(axis=1)\n    return positives.clip(upper=4)\n\n\ndef make_folds(labels: pd.DataFrame, n_folds: int = N_FOLDS,\n               seed: int = SEED) -> pd.DataFrame:\n    """Assign each study to a fold, stratified by positive-finding count.\n\n    Returns the frame with a `fold` column added.\n    """\n    frame = labels.copy().reset_index(drop=True)\n    key = stratification_key(frame)\n    rng = np.random.default_rng(seed)\n\n    frame["fold"] = -1\n    for value in key.unique():\n        rows = np.flatnonzero(key.to_numpy() == value)\n        rng.shuffle(rows)\n        frame.loc[rows, "fold"] = np.arange(len(rows)) % n_folds\n\n    if (frame["fold"] < 0).any():\n        raise RuntimeError("some studies were not assigned a fold")\n    return frame\n\n\ndef iter_folds(labels: pd.DataFrame,\n               n_folds: int = N_FOLDS) -> Iterator[Tuple[pd.DataFrame, pd.DataFrame]]:\n    """Yield (train, validation) frames for each fold."""\n    if "fold" not in labels.columns:\n        labels = make_folds(labels, n_folds=n_folds)\n\n    for fold in range(n_folds):\n        validation = labels[labels["fold"] == fold]\n        train = labels[labels["fold"] != fold]\n        yield train.reset_index(drop=True), validation.reset_index(drop=True)\n\n\ndef series_for_studies(manifest: pd.DataFrame,\n                       study_uids: Sequence[str]) -> pd.DataFrame:\n    """All cached series belonging to a set of studies."""\n    wanted = set(study_uids)\n    return manifest[manifest[config.ID_COL].isin(wanted)].reset_index(drop=True)\n\n\ndef summarise(labels: pd.DataFrame, n_folds: int = N_FOLDS) -> pd.DataFrame:\n    """Per-fold positive rates, to confirm the folds are actually comparable."""\n    if "fold" not in labels.columns:\n        labels = make_folds(labels, n_folds=n_folds)\n\n    rows = []\n    for fold in range(n_folds):\n        subset = labels[labels["fold"] == fold]\n        row = {"fold": fold, "studies": len(subset)}\n        for name in config.LABELS:\n            row[name] = float((subset[name] > 0.5).mean())\n        rows.append(row)\n    return pd.DataFrame(rows).set_index("fold")\n',
    'src/labels/__init__.py': ''
}

for path, source in FILES.items():
    target = pathlib.Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source)

sys.path.insert(0, ".")

# Drop any previously imported copy before importing. Writing the files above
# does not affect a module already in sys.modules, so re-running this cell in a
# kernel that has already imported src/ silently keeps the OLD code -- the new
# file sits on disk unused, and the run looks fine until something calls a
# function that only exists in the new version.
for _name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_name]
importlib.invalidate_caches()

from src import config
from src.data import dicom_io, cache, index as index_mod

# Only some notebooks materialise the model package.
if pathlib.Path("src/model/side_classifier.py").exists():
    from src.model import side_classifier
if pathlib.Path("src/model/net.py").exists():
    from src.model import embed, dataset as knee_dataset, net, splits

print("modules ready:", len(FILES), "files")
print("labels:", len(config.LABELS))
print("cache geometry:", config.CACHE_SLICES, "x", config.CACHE_SIZE)
print("data root:", config.DATA_ROOT)

In [ ]:
import glob
import os
import shutil

import numpy as np

SLICES_PER_SERIES = 8   # must match notebook 08


def find_inputs(pattern):
    found = sorted(glob.glob(f"/kaggle/working/{pattern}"))
    for root in sorted(glob.glob("/kaggle/input/*")):
        if os.path.basename(root) == "competitions":
            continue
        found += sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
    return found


cache_path = find_inputs("cache_train_series_*.npy")[0]
manifest_path = find_inputs("manifest_train_series_*.parquet")[0]
print("source:", cache_path, f"{os.path.getsize(cache_path) / 1e9:.1f} GB")

## Thin and write

Slice indices are the same evenly spaced selection notebook 08 makes, so the
thinned cache is exactly what it would have read from the full one.

In [ ]:
import time

volumes = np.load(cache_path, mmap_mode="r")
keep = np.linspace(0, config.CACHE_SLICES - 1,
                   SLICES_PER_SERIES).round().astype(int)
print("volumes:", volumes.shape, "-> keeping slices", keep.tolist())

out = np.zeros((volumes.shape[0], SLICES_PER_SERIES, volumes.shape[2],
                volumes.shape[3]), dtype=np.uint8)

start = time.time()
CHUNK = 256
for begin in range(0, len(volumes), CHUNK):
    end = min(begin + CHUNK, len(volumes))
    out[begin:end] = np.asarray(volumes[begin:end])[:, keep]
    if begin % (CHUNK * 8) == 0:
        print(f"  {begin}/{len(volumes)}", flush=True)

name = os.path.basename(cache_path).replace(".npy", f"_s{SLICES_PER_SERIES}.npy")
np.save(f"/kaggle/working/{name}", out)
shutil.copy(manifest_path, "/kaggle/working/")

size = os.path.getsize(f"/kaggle/working/{name}") / 1e9
print(f"\n{name}  {out.shape}  {size:.2f} GB  in {(time.time() - start) / 60:.1f} min")

## Bundle the small files

So the Colab side needs one download rather than four.

In [ ]:
for pattern in ("train_labels.parquet", "side_predictions.parquet"):
    for path in find_inputs(pattern):
        shutil.copy(path, "/kaggle/working/")
        break

for name in ("train.csv", "train_series.csv"):
    hits = sorted(glob.glob(f"/kaggle/input/competitions/*/{name}")) or \
        sorted(glob.glob(f"/kaggle/input/*/{name}"))
    if hits:
        shutil.copy(hits[0], "/kaggle/working/")

total = sum(os.path.getsize(f"/kaggle/working/{f}")
            for f in os.listdir("/kaggle/working")
            if os.path.isfile(f"/kaggle/working/{f}"))
print(f"bundle: {total / 1e9:.2f} GB")
for f in sorted(os.listdir("/kaggle/working")):
    path = f"/kaggle/working/{f}"
    if os.path.isfile(path):
        print(f"  {f}  {os.path.getsize(path) / 1e6:.1f} MB")

## Next

Save & Run All, then publish the output. On Colab:

```python
!pip install -q kaggle
# upload ~/.kaggle/access_token, then:
!kaggle kernels output <user>/<this-notebook-slug> -p data
```

Notebook 08 looks in `./data` before `/kaggle/input`, so it runs unchanged in
either place.